# Journal de bord détaillé — CISIA Santé (CIF)

**Candidat :** Léo Segalini Briant  
**Certification :** CIF — production écrite + journal de méthode/choix  
**Sujet :** prédiction du risque de réadmission hospitalière à 30 jours  
**Support de restitution :** cahier électronique Jupyter (`notebooks/`)  
**Mode de remise :** **archive ZIP** (pas de lien GitHub) — script `scripts/prepare_archive_cif.sh`

> Données **synthétiques**. Exercice pédagogique. **Jamais** une décision clinique réelle.

Ce document est le **journal de bord complet**. Pour chaque étape, on documente :  
**décision → justification → alternative écartée → preuve** dans le cahier ou le code.

La **production écrite exécutable** reste le parcours technique (inventaire → réponse). Ce journal **relie** tous les choix.


## Mode d’emploi du rendu (archive ZIP uniquement)

1. Générer l’archive minimale : `./scripts/prepare_archive_cif.sh`  
2. Remettre **uniquement** `dist/CISIA_Sante_CIF_SegaliniBriant_YYYYMMDD.zip` (pas de lien GitHub).  
3. L’archive **n’inclut pas** : PDF sujet, PPTX, DOCX, notes orales, livrables nominatifs, `.env`, GGUF, `.joblib`.  
4. Le destinataire dézippe → `LIRE_MOI_RENDU_CIF.md` → Jupyter → ce journal puis le parcours.

**Dans Jupyter :** Run All sur ce notebook, puis suivre [00_sommaire.ipynb](00_sommaire.ipynb).


In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import os
import sys

from IPython.display import Markdown, display
import pandas as pd

# Colab : se placer dans le projet dézippé si besoin
for _colab in (Path("/content/CISIA_Sante"), Path("/content/CISIA - Sujet Santé")):
    if (_colab / "Sujet.md").is_file() and (
        (_colab / "donnees" / "patients.csv").is_file() or (_colab / "patients.csv").is_file()
    ):
        os.chdir(_colab)
        break


def _est_racine(p: Path) -> bool:
    if not p.is_dir() or not (p / "Sujet.md").is_file():
        return False
    return (p / "donnees" / "patients.csv").is_file() or (p / "patients.csv").is_file()


_candidates: list[Path] = []
if os.environ.get("CISIA_ROOT"):
    _candidates.append(Path(os.environ["CISIA_ROOT"]))
_candidates.extend([Path.cwd(), *Path.cwd().parents])
if Path("/content").is_dir():
    _candidates.extend([Path("/content/CISIA_Sante"), Path("/content")])
    _candidates.extend(p for p in Path("/content").iterdir() if p.is_dir())

try:
    ROOT = next(p.resolve() for p in _candidates if _est_racine(p))
except StopIteration as exc:
    raise FileNotFoundError(
        "Racine CISIA introuvable (Sujet.md + donnees/patients.csv). "
        "Dézippez l'archive et ouvrez le notebook depuis le projet, "
        "ou définissez CISIA_ROOT."
    ) from exc

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(ROOT / "notebooks") not in sys.path:
    sys.path.insert(0, str(ROOT / "notebooks"))

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 140)

NB = ROOT / "notebooks"
parcours = [
    ("00_guide/02_journal_de_bord.ipynb", "Journal de bord (ce fichier)"),
    ("00_guide/01_cadre_projet.ipynb", "Cadre & objectifs"),
    ("01_donnees/01_inventaire_sources.ipynb", "Inventaire CSV"),
    ("01_donnees/02_qualite_hallucinations.ipynb", "Qualité / hallucinations"),
    ("01_donnees/03_registre_rgpd.ipynb", "Registre RGPD"),
    ("01_donnees/05_analyse_donnees.ipynb", "Analyse qualité complète"),
    ("01_donnees/04_pipeline_donnees.ipynb", "Pipeline curated"),
    ("02_modeles/02_nettoyage_donnees.ipynb", "Nettoyage + cohorte"),
    ("02_modeles/03_preparation_modele.ipynb", "Features sortie / télé"),
    ("02_modeles/04_entrainement_modele.ipynb", "Entraînement Optuna + choix algo"),
    ("02_modeles/05_lancement_modele.ipynb", "Inférence"),
    ("02_modeles/06_reponse_modele.ipynb", "Explication / réponse"),
    ("02_modeles/01_benchmark_optuna.ipynb", "Benchmark détaillé (complément)"),
    ("03_llm/01_llm_local_cr.ipynb", "LLM local (optionnel)"),
]
rows = [
    {
        "n": i,
        "étape": titre,
        "fichier": rel,
        "présent": "oui" if (NB / rel).exists() else "NON",
    }
    for i, (rel, titre) in enumerate(parcours, 1)
]
display(Markdown("### Inventaire du cahier électronique"))
display(pd.DataFrame(rows))
print("Racine projet :", ROOT)
print("Sources CSV :", ROOT / "donnees")


## 1. Lecture du sujet et formulation du problème

### Choix 1.1 — Objectif métier retenu

| | |
|---|---|
| **Décision** | Construire un **score de risque de réadmission à 30 jours** pour **prioriser** les sorties à suivi renforcé. |
| **Pourquoi** | Le sujet pose explicitement la pression sur les capacités, le vieillissement et les réadmissions évitables. |
| **Écarté** | Un simple dashboard descriptif sans score ; un diagnostic médical automatisé. |
| **Preuve** | `Sujet.md` · `01_cadre_projet.ipynb` |

### Choix 1.2 — Double score (sortie / télé)

| | |
|---|---|
| **Décision** | Deux jeux de features : **score_sortie** (à la sortie) et **score_tele** (fenêtre post-sortie objets connectés). |
| **Pourquoi** | Respecter l’**anti-fuite temporelle** : les capteurs post-sortie ne doivent pas « prédire » une décision de sortie. |
| **Écarté** | Un modèle unique mélangeant tout l’historique y compris post-sortie. |
| **Preuve** | `features_sortie.py` / `features_tele.py` · notebooks préparation |

### Choix 1.3 — Positionnement produit

| | |
|---|---|
| **Décision** | Aide à la **priorisation** + gouvernance (registre, biais), 100 % **local**. |
| **Pourquoi** | Secret médical, RGPD art. 9, exigence de traçabilité des alertes. |
| **Écarté** | API cloud externe sur données nominatives ; score opaque sans explication. |
| **Preuve** | Webapp `/cisia` · registre · journal §7 |


## 2. Organisation du cahier et de l’ingénierie

### Choix 2.1 — Cahier = dossier `notebooks/`

| | |
|---|---|
| **Décision** | Restituer la production dans Jupyter, classée `00_guide` / `01_donnees` / `02_modeles` / `03_llm`. |
| **Pourquoi** | Exigence CIF « cahier électronique type Jupyter ou Kaggle » ; reproductibilité cellule par cellule. |
| **Écarté** | Un seul PDF Word figé sans code exécutable. |
| **Preuve** | Arborescence `notebooks/` · `notebooks/README.md` |

### Choix 2.2 — Code factorisé dans `src/`

| | |
|---|---|
| **Décision** | Pipeline, qualité, modèles et web dans `src/` ; les notebooks **appellent** ce code. |
| **Pourquoi** | Éviter la duplication, garantir que la démo webapp et le cahier partagent la même logique. |
| **Écarté** | Tout coller en cellules Jupyter non réutilisables. |
| **Preuve** | `src/data/`, `src/model/`, `src/web/` |

### Choix 2.3 — Registry déclaratif des sources et étapes

| | |
|---|---|
| **Décision** | `registry.py` + `pipeline_steps.py` pour lister sources CSV et étapes pipeline. |
| **Pourquoi** | Le sujet demande une solution **évolutive** à l’ajout de nouvelles sources. |
| **Écarté** | Scripts figés à 11 `if fichier == …` non extensibles. |
| **Preuve** | `src/data/registry.py` · `describe_pipeline()` |

### Choix 2.4 — Remise par archive ZIP (pas de lien)

| | |
|---|---|
| **Décision** | Script `prepare_archive_cif.sh` produisant un ZIP autonome. |
| **Pourquoi** | Contrainte de rendu : fournir une **archive**, pas une URL. |
| **Écarté** | Remise uniquement via GitHub. |
| **Preuve** | `scripts/prepare_archive_cif.sh` · `LIRE_MOI_RENDU_CIF.md` (dans le ZIP) |


## 3. Données — inventaire, cohorte, identité, unité

### Choix 3.1 — Utiliser les 11 CSV du sujet dans `donnees/` (immuables)

| | |
|---|---|
| **Décision** | Travailler sur le jeu fourni, stocké dans `donnees/` (patients, séjours, historique, diagnostics, actes, biologies, signes vitaux, médicaments, CR, objets connectés, territoire INSEE). Les CSV sources **ne sont jamais modifiés** : le pipeline les **duplique** vers `data/raw/` puis produit `data/curated/`. |
| **Pourquoi** | Jeu officiel du sujet ; séparation claire source / données de travail (train–val–test) ; reproductibilité du rendu ZIP. |
| **Écarté** | Modifier les CSV d’origine à la racine ; remplacer par un autre open data hors sujet. |
| **Preuve** | `donnees/` · `src/data/load.py` (`copy_sources_to_raw`) · `01_inventaire_sources.ipynb` |

### Choix 3.2 — Cohorte d’entraînement = sorties à domicile uniquement

| | |
|---|---|
| **Décision** | Filtre `ModeSortie == Domicile` ; **exclusion des décès** (et transferts / EHPAD / HAD). |
| **Pourquoi** | Sens clinique : on ne prédit pas une réadmission après un décès ; évite une fuite absurde. |
| **Écarté** | Entraîner sur 100 % des séjours sans filtre de mode de sortie. |
| **Preuve** | `src/data/cohort.py` · cellule ci-dessous · page `/cisia/analyse` |

### Choix 3.3 — Coffre identité / pseudonymisation

| | |
|---|---|
| **Décision** | Isoler le nominatif (`vault_identite`) ; features sans PII directe. |
| **Pourquoi** | RGPD art. 9 + secret médical ; principe du moindre privilège. |
| **Écarté** | Garder NomPrenom dans les matrices ML. |
| **Preuve** | `src/data/identity.py` · `03_registre_rgpd.ipynb` |

### Choix 3.4 — Exclusion des proxys socio-individuels hors périmètre

| | |
|---|---|
| **Décision** | Ne pas injecter de variables socio-économiques individuelles non collectées à l’hôpital ; territoire = **agrégats** INSEE seulement. |
| **Pourquoi** | Exigence explicite du sujet. |
| **Écarté** | Enrichir avec revenus individuels ou scores sociaux inventés. |
| **Preuve** | Registre d’usage · `territoire_insee.csv` via jointure agrégée |

### Choix 3.5 — Unités des constantes

| | |
|---|---|
| **Décision** | **Une unité canonique par mesure** (mmHg, °C, %, /min, kg). Pas de mélange °C/°F dans une même colonne. Pas de conversion de toutes les constantes vers une « unité unique ». |
| **Pourquoi** | Le modèle apprend des colonnes distinctes ; mélanger des unités pour une même variable fausse l’apprentissage. |
| **Écarté** | Tout mettre en SI unique artificiel ; laisser des unités hétérogènes non documentées. |
| **Preuve** | Table ci-dessous · `choix_algorithmes.unites_constantes` |


In [ ]:
# Si la cellule d'init a échoué / a été sautée : retrouver la racine (donnees/)
if "ROOT" not in globals():
    from pathlib import Path as _P
    import os as _os, sys as _sys
    def _est(_p):
        return _p.is_dir() and (_p / "Sujet.md").is_file() and (
            (_p / "donnees" / "patients.csv").is_file() or (_p / "patients.csv").is_file()
        )
    _cands = [_P.cwd(), *_P.cwd().parents]
    if _os.environ.get("CISIA_ROOT"):
        _cands.insert(0, _P(_os.environ["CISIA_ROOT"]))
    ROOT = next(_p.resolve() for _p in _cands if _est(_p))
    _os.chdir(ROOT)
    for _s in (str(ROOT), str(ROOT / "notebooks")):
        if _s not in _sys.path:
            _sys.path.insert(0, _s)

from src.data.paths import ProjectPaths
from src.data.load import load_csv
from src.data.analyse_donnees import cohort_training_summary, build_analyse_complete
from src.model.choix_algorithmes import contenu_choix_modeles

paths = ProjectPaths(root=ROOT)
c = contenu_choix_modeles()

sej = load_csv(paths, "sejours.csv", from_raw=False)
coh = cohort_training_summary(sej)
display(Markdown("#### Preuve — cohorte"))
display(Markdown(
    f"- **Règle :** {coh['regle']}  \n"
    f"- Séjours bruts : **{coh['n_sejours_bruts']}**  \n"
    f"- Éligibles entraînement : **{coh['n_eligibles_entrainement']}**  \n"
    f"- Exclus : **{coh['n_exclus']}** (dont décès : **{coh['n_deces_exclus']}**)"
))
display(pd.DataFrame(
    [{"ModeSortie": k, "N": v, "dans_train": k == "Domicile"}
     for k, v in coh["repartition_mode_sortie"].items()]
))

display(Markdown("#### Preuve — unités canoniques"))
display(pd.DataFrame(c["unites_constantes"]["table"]))
display(Markdown(c["unites_constantes"]["resume"]))


## 4. Qualité des données — avant le modèle

### Choix 4.1 — Audit systématique (trous, aberrants, mal notés, temporalité)

| | |
|---|---|
| **Décision** | Produire un rapport qualité + page/notebook d’analyse (nulls, sentinelles, dates hors séjour, patho↔retour). |
| **Pourquoi** | Le sujet insiste sur l’hétérogénéité et la fiabilité variable des sources cliniques. |
| **Écarté** | Entraîner directement sur CSV bruts sans diagnostic. |
| **Preuve** | `02_qualite_hallucinations.ipynb` · `05_analyse_donnees.ipynb` · `/cisia/analyse` |

### Choix 4.2 — Traiter les anomalies synthétiques comme matériel pédagogique

| | |
|---|---|
| **Décision** | Documenter les incohérences volontaires (ex. actes hors fenêtre de séjour) plutôt que les cacher. |
| **Pourquoi** | Montrer la capacité d’audit ; aligné avec l’esprit « hallucinations / mal notés » du parcours. |
| **Écarté** | Nettoyer silencieusement sans journaliser. |
| **Preuve** | Flags `quality_report` · graphiques analyse |

### Choix 4.3 — Ce qui n’est pas fiable n’entre pas silencieusement dans le score

| | |
|---|---|
| **Décision** | Règles de correction / exclusion (durée incohérente, signaux OC non « Bon », etc.) dans le pipeline. |
| **Pourquoi** | Réduire les fausses alertes et la contamination du modèle. |
| **Écarté** | Garder toutes les lignes « telles quelles » pour maximiser N artificiellement. |
| **Preuve** | `src/data/quality.py` · `vitals.py` · étapes pipeline |


In [ ]:
analyse = build_analyse_complete(paths)
sq = analyse["stats_qualite"]
display(Markdown("#### Preuve — synthèse qualité (effectifs)"))
display(pd.DataFrame(sq["categories"]))
display(Markdown("#### Preuve — fichiers avec anomalies détectées"))
par = pd.DataFrame(sq["par_fichier"])
display(par.loc[(par["n_mal_notes"] + par["n_aberrants"]) > 0].sort_values(
    "n_mal_notes", ascending=False
))


## 5. Split, anti-fuite, préparation des features

### Choix 5.1 — Split patient-level 70 % / 15 % / 15 %

| | |
|---|---|
| **Décision** | Train / validation / test au niveau **PatientID** (ratios cibles 0,70 / 0,15 / 0,15). |
| **Pourquoi** | Interdit d’entraîner et tester sur les mêmes patients ; validation pour seuil F2 et Optuna ; test pour mesure finale. **Ce n’est ni 100 % train+100 % test, ni un simple 70/30 sans validation.** |
| **Écarté** | Split aléatoire par ligne de séjour (fuite patient) ; tout entraîner puis « tester » sur le même jeu. |
| **Preuve** | `src/data/cohort.py` · colonne `split` des parquets |

### Choix 5.2 — Features sortie sans objets connectés post-sortie

| | |
|---|---|
| **Décision** | Score sortie : informations disponibles **à la sortie** (séjour, bio, vitaux hospitaliers, médication, patho, territoire agrégé…). |
| **Pourquoi** | Anti-fuite : on ne peut pas utiliser le futur pour décider à J0. |
| **Écarté** | Inclure `oc_*` post-sortie dans score_sortie. |
| **Preuve** | `features_sortie.py` · notebook préparation |

### Choix 5.3 — Features télé sur fenêtre post-sortie (signal Bon)

| | |
|---|---|
| **Décision** | Agrégats objets connectés sur une fenêtre fixe après sortie, filtrés `QualiteSignal == Bon` (dont poids). |
| **Pourquoi** | Cas d’usage télésuivi distinct ; qualité de signal. |
| **Écarté** | Mélanger télé et sortie dans un seul label temporel. |
| **Preuve** | `features_tele.py` |

### Choix 5.4 — Encodage pathologies chroniques

| | |
|---|---|
| **Décision** | Flags `patho_*` + compte `n_pathologies` à partir du champ multi-valeurs. |
| **Pourquoi** | Analyser le lien pathologie ↔ retour 30 j ; features exploitables. |
| **Écarté** | Ignorer les comorbidités déclarées. |
| **Preuve** | `aggregate_pathologies` · analyse pathologies |


In [ ]:
feat = paths.curated / "features_score_sortie.parquet"
if feat.exists():
    df = pd.read_parquet(feat)
    display(Markdown("#### Preuve — split observé (score sortie)"))
    vc = df["split"].value_counts()
    display(pd.DataFrame({
        "split": vc.index.astype(str),
        "n_sejours": vc.values,
        "pct": (100.0 * vc / len(df)).round(1).values,
        "n_patients": [df.loc[df["split"] == s, "PatientID"].nunique() for s in vc.index],
    }))
    display(Markdown(c["protocole_split"]["resume"]))
    display(Markdown(c["protocole_split"]["pas_70_30_simple"]))
else:
    display(Markdown(
        "> **Parquet absent.** Dans l’archive dézippée, lancer :  \n"
        "> `PYTHONPATH=. python scripts/run_pipeline.py` puis ré-exécuter cette cellule."
    ))


### Choix 6.4 — Random Forest pour score_sortie (production)

| | |
|---|---|
| **Décision** | Retenir **random_forest** pour le score sortie (`PRODUCTION_MODELS`). |
| **Pourquoi** | Critère = PR-AUC test ; RF 1ᵉʳ sur ce run (~0,58). Vs LightGBM : F2 LGBM plus haut mais PR-AUC plus bas ; RF = SHAP + sklearn + stabilité petit n. Même si un seed place LGBM devant de peu, on fige RF (écart négligeable). |
| **Écarté en prod** | LightGBM (benchmarké, souvent meilleur F2) ; HistGB ; MLP (chute relative). |
| **Preuve** | `models/score_sortie_metrics.json` · graphique `documentation/assets/comparaison_modeles_rf.png` (RETENU vs 1ᵉʳ PR-AUC) |


In [ ]:
display(Markdown("#### Preuve — décision RF vs HistGB (extrait structuré)"))
display(Markdown(c["contexte_n"]))
display(Markdown(
    f"**Retenu :** {c['decision_actuelle']['libelle']} — {c['decision_actuelle']['pourquoi']}"
))
display(Markdown(f"**Critère :** {c['decision_actuelle']['critere']}"))

av_rf = pd.DataFrame({"Random Forest — avantages": c["rf"]["avantages"]})
in_rf = pd.DataFrame({"Random Forest — inconvénients": c["rf"]["inconvenients"]})
av_h = pd.DataFrame({"HistGB — avantages": c["histgb_detail"]["avantages"]})
in_h = pd.DataFrame({"HistGB — inconvénients": c["histgb_detail"]["inconvenients"]})
display(av_rf)
display(in_rf)
display(Markdown(c["histgb"]["pour_grand_jeu"]))
display(av_h)
display(in_h)

mpath = ROOT / "models" / "score_sortie_metrics.json"
if mpath.exists():
    m = json.loads(mpath.read_text(encoding="utf-8"))
    rows = []
    for key in m.get("ranking_test_pr_auc", []):
        block = m.get(key)
        if not isinstance(block, dict) or "test" not in block:
            continue
        t = block["test"]
        rows.append({
            "rang_PR-AUC": len(rows) + 1,
            "modele": key,
            "PR-AUC test": round(float(t["pr_auc"]), 4),
            "F2 test": round(float(t["f2"]), 4),
            "rappel": round(float(t["recall"]), 3),
            "précision": round(float(t["precision"]), 3),
            "seuil": round(float(t["threshold"]), 3),
        })
    display(Markdown(f"#### Preuve — ranking test (`retenu` = `{m.get('retenu')}`)"))
    display(pd.DataFrame(rows))
else:
    display(Markdown("_metrics absentes — lancer l’entraînement dans le cahier._"))

man = ROOT / "models" / "production_manifest.json"
if man.exists():
    mf = json.loads(man.read_text(encoding="utf-8"))
    display(Markdown("#### Preuve — manifeste production"))
    display(pd.DataFrame([
        {
            "score": k,
            "modèle": v.get("retenu"),
            "seuil": v.get("threshold"),
            "PR-AUC test": round(float(v.get("test_pr_auc", 0)), 4),
            "F2 test": round(float(v.get("test_f2", 0)), 4),
            "calibré": v.get("calibrated"),
        }
        for k, v in (mf.get("scores") or {}).items()
    ]))


## 7. Explicabilité, LLM, webapp, éthique

### Choix 7.1 — SHAP + texte métier

| | |
|---|---|
| **Décision** | Expliquer chaque score (facteurs locaux + phrase). |
| **Pourquoi** | Confiance soignant/jury ; pas de boîte noire. |
| **Écarté** | Afficher seulement un %. |
| **Preuve** | `explain_patient.py` · `06_reponse_modele.ipynb` · fiche `/cisia/sejours/{id}` |

### Choix 7.2 — LLM local distinct du score

| | |
|---|---|
| **Décision** | LLM (Qwen GGUF) pour brief sur CR **masqué**, exécution locale ; **ne calcule pas** le risque. |
| **Pourquoi** | Séparer clairement modèle tabulaire et génération de texte ; souveraineté. |
| **Écarté** | Envoyer les CR nominatifs à une API cloud ; faire générer le % par le LLM. |
| **Preuve** | `03_llm/01_llm_local_cr.ipynb` |

### Choix 7.3 — Deux modules UI (Readmit / Flow)

| | |
|---|---|
| **Décision** | `/cisia` = gouvernance & score ; `/soignant` = plan d’étage / constantes. |
| **Pourquoi** | Montrer l’opérationnalisation du sujet (intégration SIH / terrain). |
| **Écarté** | Une seule UI confuse mélangeant jury et lits. |
| **Preuve** | Webapp · slide « deux modules » |

### Choix 7.4 — Monitoring continu vs saisie IDE + poids

| | |
|---|---|
| **Décision** | Soins continus = machines live ; autres services = saisie manuelle (dont **PoidsKg**). |
| **Pourquoi** | Réalisme terrain ; le poids existait déjà en OC, pas en saisie IDE. |
| **Écarté** | Forcer le live partout. |
| **Preuve** | `live.py` · `vitals_saisie.js` |

### Choix 7.5 — Audit des biais exposé

| | |
|---|---|
| **Décision** | Page `/cisia/biais` (âge, sexe, territoire). |
| **Pourquoi** | « On mesure, on ne cache pas » — exigence éthique du sujet. |
| **Écarté** | Publier un unique score global sans stratification. |
| **Preuve** | `cisia_biais` · notebook / exports biais |

### Choix 7.6 — Place de la webapp dans le rendu CIF

| | |
|---|---|
| **Décision** | Inclure le code webapp dans l’archive comme **démonstration**, le **cahier Jupyter** restant le livrable écrit obligatoire. |
| **Pourquoi** | Le sujet demande une solution opérationnelle *et* une production écrite notebook. |
| **Écarté** | Remplacer le journal par des captures d’écran seules. |
| **Preuve** | Ce journal · `LIRE_MOI_RENDU_CIF.md` |


## 8. Tableau récapitulatif de tous les choix majeurs


In [ ]:
recap = pd.DataFrame([
    {"id": "1.1", "thème": "Objectif", "décision": "Score priorisation réadmission 30 j", "écarté": "Diagnostic médical auto"},
    {"id": "1.2", "thème": "Features", "décision": "Deux scores sortie / télé", "écarté": "Modèle unique avec fuite temporelle"},
    {"id": "1.3", "thème": "Produit", "décision": "Local + explicable + gouvernable", "écarté": "Cloud opaque"},
    {"id": "2.1", "thème": "Livrable", "décision": "Cahier Jupyter notebooks/", "écarté": "PDF seul"},
    {"id": "2.2", "thème": "Architecture", "décision": "Logique dans src/", "écarté": "Code seulement dans cellules"},
    {"id": "2.3", "thème": "Évolutivité", "décision": "Registry sources/étapes", "écarté": "Pipeline figé non extensible"},
    {"id": "2.4", "thème": "Remise", "décision": "Archive ZIP", "écarté": "Lien GitHub seul"},
    {"id": "3.1", "thème": "Données", "décision": "11 CSV dans donnees/ (copie → data/raw)", "écarté": "Modifier les CSV sources / autre dataset"},
    {"id": "3.2", "thème": "Cohorte", "décision": "Domicile only (décès exclus)", "écarté": "Tous ModeSortie"},
    {"id": "3.3", "thème": "RGPD", "décision": "Coffre identité", "écarté": "PII dans le modèle"},
    {"id": "3.4", "thème": "Éthique data", "décision": "Pas de socio individuel", "écarté": "Proxys hors périmètre"},
    {"id": "3.5", "thème": "Unités", "décision": "Canonique par mesure", "écarté": "Mélange d’unités / unité unique artificielle"},
    {"id": "4.1", "thème": "Qualité", "décision": "Audit avant modèle", "écarté": "Train sur brut"},
    {"id": "4.2", "thème": "Anomalies", "décision": "Documenter hallucinations", "écarté": "Nettoyage silencieux"},
    {"id": "4.3", "thème": "Fiabilité", "décision": "Exclure / corriger le non fiable", "écarté": "Tout garder"},
    {"id": "5.1", "thème": "Split", "décision": "70/15/15 patient-level", "écarté": "100% train+test ; split ligne"},
    {"id": "5.2", "thème": "Anti-fuite", "décision": "Pas d’OC post-sortie dans score sortie", "écarté": "Mélanger futur"},
    {"id": "5.3", "thème": "Télé", "décision": "Fenêtre post-sortie signal Bon", "écarté": "Tous signaux"},
    {"id": "5.4", "thème": "Patho", "décision": "Flags patho_*", "écarté": "Ignorer comorbidités"},
    {"id": "6.1", "thème": "Benchmark", "décision": "Multi-modèles + baseline", "écarté": "Un seul algo"},
    {"id": "6.2", "thème": "Métrique", "décision": "PR-AUC test", "écarté": "Accuracy"},
    {"id": "6.3", "thème": "Seuil", "décision": "F2 max sur validation", "écarté": "Seuil 0,5 ou calé sur test"},
    {"id": "6.4", "thème": "Prod sortie", "décision": "Random Forest", "écarté": "MLP / HistGB par défaut"},
    {"id": "6.5", "thème": "Prod télé", "décision": "Logistic", "écarté": "RF partout"},
    {"id": "6.6", "thème": "Futur", "décision": "HistGB documenté grand N", "écarté": "Forcer boosting petit N"},
    {"id": "6.7", "thème": "Tuning", "décision": "Optuna sur val", "écarté": "Tuning sur test"},
    {"id": "6.8", "thème": "Calibration", "décision": "Sigmoid prod", "écarté": "Probas brutes"},
    {"id": "7.1", "thème": "XAI", "décision": "SHAP + texte", "écarté": "% seul"},
    {"id": "7.2", "thème": "LLM", "décision": "Local, hors score", "écarté": "Cloud / LLM=score"},
    {"id": "7.3", "thème": "UI", "décision": "Readmit + Flow", "écarté": "UI unique confuse"},
    {"id": "7.4", "thème": "Vitals", "décision": "Live vs saisie + poids", "écarté": "Live partout"},
    {"id": "7.5", "thème": "Biais", "décision": "Audit exposé", "écarté": "Score unique non stratifié"},
    {"id": "7.6", "thème": "CIF", "décision": "Notebook = écrit ; web = démo", "écarté": "Captures seules"},
])
display(Markdown("### Synthèse journal — 33 choix structurants"))
display(recap)


## 9. Limites assumées (transparence)

1. **Données synthétiques** et effectif limité (~480 séjours éligibles) : les performances ne se généralisent pas à un CHU réel.  
2. Les métriques dépendent du seed de split et de la calibration.  
3. Le LLM local peut être absent (poids GGUF non inclus dans l’archive pour la taille).  
4. HistGB n’est pas le bundle de production actuel : choix **volontaire** lié au volume.  
5. La webapp illustre l’intégration ; le **cahier Jupyter + ce journal** constituent le cœur du rendu écrit CIF.


## 10. Procédure concrète pour fournir l’archive

Sur la machine de préparation du rendu :

```bash
cd "/chemin/vers/CISIA - Sujet Santé"
chmod +x scripts/prepare_archive_cif.sh
./scripts/prepare_archive_cif.sh
```

Résultat : `dist/CISIA_Sante_CIF_SegaliniBriant_YYYYMMDD.zip`

**À remettre :** uniquement ce ZIP.  
**Ne pas remettre :** lien GitHub comme seul livrable ; `.env` ; `.venv` ; PowerPoint personnel (hors cahier).

Après dézippage par le correcteur : suivre `LIRE_MOI_RENDU_CIF.md`, ouvrir Jupyter, exécuter **ce journal (Run All)** puis le parcours du sommaire.

---

*Fin du journal de bord détaillé.*
